# Laboratório: MCP

**Dupla:**

- Nome 1: José Cássio
- Nome 2: Felipe Baraúna

Neste laboratório deverão ser construídos um servidor MCP e uma skill para um domínio escolhido pela dupla. O servidor é usado por um agente que também se conecta a pelo menos um servidor MCP da comunidade, e a skill ensina esse agente a fazer uma tarefa do domínio com as ferramentas do servidor. O domínio pode ser o mesmo do laboratório de RAG ou do agente feito antes, e nesse caso o retriever ou as ferramentas construídas lá viram ferramentas do servidor, ou pode ser um domínio novo. Cada seção descreve o que se espera dela e traz um espaço para o código, e podem ser criadas quantas células forem precisas.

Um servidor MCP pode ser usado por qualquer cliente, com modelos e mensagens de sistema que a dupla não controla. Tudo o que um agente sabe sobre o servidor vem dos nomes, das docstrings e dos parâmetros, então essas descrições devem ser escritas para quem nunca viu o código.

In [1]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph "langchain[mcp]" uv

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [2]:
import os
import sys
from pathlib import Path

os.environ["FASTMCP_LOG_LEVEL"] = "WARNING"

import pandas as pd
import yaml

from fastmcp import Client
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.mcp import MCPAdapter
from langchain.messages import HumanMessage
from langchain.tools import tool

/tmp/ipykernel_229915/1198791395.py:13: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


In [3]:
import json
import shutil

from dotenv import load_dotenv
from fastmcp.exceptions import ToolError
from langchain.messages import AIMessage, ToolMessage

load_dotenv("../.env")
pd.set_option("display.max_colwidth", 160)
model = init_chat_model(os.environ.get("MODELO", "google_genai:gemini-3.5-flash-lite"), temperature=0)

## Servidor

O servidor deve ser escrito com o `FastMCP` em um arquivo `server.py`, criado pelo grupo na mesma pasta do notebook e entregue junto com ele. Ele deve oferecer:

- **Ferramentas:** o retriever do RAG ou as ferramentas do agente anterior, se o domínio for o mesmo, e pelo menos duas outras ferramentas, que podem consultar uma API pública, calcular algo ou ler dados próprios do domínio.
- **Recurso:** pelo menos um, com URI fixa ou template, com dados que a aplicação anexa ao contexto.
- **Prompt:** pelo menos um, com argumentos.

O arquivo segue este formato:

```python
from fastmcp import FastMCP

mcp = FastMCP("nome-do-servidor")

# Ferramentas, recursos e prompts aqui.

if __name__ == "__main__":
    mcp.run(show_banner=False)
```

O servidor usa o transporte stdio: o cliente inicia o `server.py` como um processo separado a cada conexão e conversa com ele pela entrada e saída padrão, sem porta nem URL. Por ser outro processo, o servidor não enxerga as variáveis definidas aqui, então tudo de que ele precisa, como o modelo de embedding e o caminho do índice, deve ser criado dentro do arquivo.

**Domínio:** o mesmo do laboratório de RAG e do nosso projeto da unidade, o Assistente de Bancada: ligar componentes num ESP32. O servidor `bancada` (`server.py`) oferece:

- **Ferramentas:** `buscar_datasheet`, o retriever do laboratório de RAG (mesmo índice e mesmo embedding), e três cálculos que vêm do servidor MCP do nosso projeto: `resistor_led`, `divisor_tensao` e `largura_trilha`.
- **Recursos:** `esp32://restricoes-gpio` (URI fixa, com as restrições dos GPIOs tiradas do datasheet, para a aplicação anexar ao contexto) e o template `datasheet://{documento}/{pagina}`, que devolve a página inteira de um datasheet a partir do índice.
- **Prompt:** `ligar_componente(componente, pino)`, que monta o pedido de conferência de uma ligação.

Os argumentos inválidos levantam `ToolError` com uma mensagem que diz o que corrigir, para o agente conseguir tentar de novo.

### Índice do RAG

Se o servidor usa o retriever, o índice do vector store precisa existir na pasta em que o servidor roda. No Colab, a pasta do índice pode ser enviada compactada e descompactada aqui, ou o índice pode ser refeito com o código de indexação do laboratório de RAG. O modelo de embedding do servidor deve ser o mesmo usado para criar o índice.

In [4]:
# O servidor usa o índice do laboratório de RAG (coleção "trechos-1000"). Aqui ele é copiado para esta pasta;
# no Colab, envie a pasta indice/ compactada junto com o notebook. O embedding (multilingual-e5-small) é
# criado dentro do server.py e baixado para modelos/ na primeira execução.
if not Path("indice").exists():
    shutil.copytree("../lab01-rag/indice", "indice")
print(sorted(p.name for p in Path("indice").iterdir())[:5])

['0e97243b-ddbd-4039-8b57-9ee14df4c51e', '6affa710-e6aa-4532-bfe9-9b092fa92e90', 'chroma.sqlite3']


### Teste do servidor

O servidor deve ser testado com um cliente que liste as ferramentas, os recursos e os prompts e que chame cada ferramenta pelo menos uma vez, incluindo um caso com argumento inválido. O `MCPAdapter` converte só as ferramentas; os recursos e os prompts são listados e lidos com o `Client` do FastMCP. Os dois recebem o caminho do `server.py` e iniciam o servidor sozinhos, e um erro no arquivo aparece como exceção na célula que abre a conexão.

In [5]:
SERVER = Path("server.py")

async with MCPAdapter(SERVER) as adapter:
    own_tools = await adapter.list_tools()

pd.DataFrame([{"name": tool.name, "description": tool.description, "args": list(tool.args)} for tool in own_tools])

,name,description,args
0,buscar_datasheet,"Busca trechos nos datasheets indexados e devolve o texto com documento e página para citar.\n\nUse para qualquer valor técnico (tensão, corrente, pino, temp...","[consulta, documento, k]"
1,resistor_led,"Calcula o resistor em série de um LED, R = (tensão da fonte - queda do LED) / corrente.\n\ntensao_fonte: tensão que alimenta o LED, em volts (num GPIO do ES...","[tensao_fonte, tensao_led, corrente_ma]"
2,divisor_tensao,"Dimensiona um divisor resistivo para baixar um sinal, Vsaída = Ventrada * R2 / (R1 + R2).\n\nUse quando um sinal de 5 V precisa entrar num GPIO de 3,3 V (ex...","[tensao_entrada, tensao_saida, r2_ohms]"
3,largura_trilha,"Largura mínima de uma trilha de placa de circuito impresso pela norma IPC-2221.\n\ncorrente_a: corrente na trilha, em ampères. espessura_cobre_oz: cobre em ...","[corrente_a, espessura_cobre_oz, elevacao_temperatura_c, camada]"


In [6]:
def mostrar_resultado(r):
    return r.data if r.data is not None else [c.text for c in r.content]


async with Client(SERVER) as client:
    recursos = await client.list_resources()
    templates = await client.list_resource_templates()
    prompts = await client.list_prompts()
    print("Recursos:", [str(r.uri) for r in recursos])
    print("Templates:", [t.uriTemplate for t in templates])
    print("Prompts:", [(p.name, [a.name for a in p.arguments]) for p in prompts], "\n")

    chamadas = [
        ("buscar_datasheet", {"consulta": "working voltage working current", "documento": "hc-sr04", "k": 2}),
        ("resistor_led", {"tensao_fonte": 3.3, "tensao_led": 2.0, "corrente_ma": 10}),
        ("divisor_tensao", {"tensao_entrada": 5, "tensao_saida": 3.3}),
        ("largura_trilha", {"corrente_a": 1.0}),
        ("buscar_datasheet", {"consulta": "pinout", "documento": "arduino-uno"}),
        ("resistor_led", {"tensao_fonte": 1.5, "tensao_led": 2.0}),
    ]
    for nome, args in chamadas:
        try:
            r = await client.call_tool(nome, args)
            texto = json.dumps(mostrar_resultado(r), ensure_ascii=False)
            print(f"{nome}({args})\n  -> {texto[:300]}\n")
        except ToolError as e:
            print(f"{nome}({args})\n  -> erro: {e}\n")

    print((await client.read_resource("esp32://restricoes-gpio"))[0].text)
    print((await client.read_resource("datasheet://hc-sr04/1"))[0].text[:400], "\n")
    prompt = await client.get_prompt("ligar_componente", {"componente": "HC-SR04", "pino": "GPIO18"})
    print(prompt.messages[0].content.text)

Recursos: ['esp32://restricoes-gpio']
Templates: ['datasheet://{documento}/{pagina}']
Prompts: [('ligar_componente', ['componente', 'pino'])] 



/tmp/ipykernel_229915/1788113231.py:10: FastMCPDeprecationWarning: Accessing `ResourceTemplate.uriTemplate` is deprecated; MCP SDK v2 renamed this field to `uri_template`. Update your code to read `.uri_template` instead.
  print("Templates:", [t.uriTemplate for t in templates])


buscar_datasheet({'consulta': 'working voltage working current', 'documento': 'hc-sr04', 'k': 2})
  -> [{"documento": "hc-sr04", "titulo": "HC-SR04 Ultrasonic Ranging Module", "pagina": 1, "secao": null, "distancia": 0.169, "texto": "Working Frequency 40Hz\nMax Range 4m\nMin Range 2cm\nMeasuringAngle 15 degree\nTrigger Input Signal 10uS TTL pulse\nEcho Output Signal Input TTL lever signal and the ran

resistor_led({'tensao_fonte': 3.3, 'tensao_led': 2.0, 'corrente_ma': 10})
  -> {"resistor_ideal_ohms": 130.0, "resistor_comercial_ohms": 150.0, "corrente_real_ma": 8.67, "potencia_mw": 11.3}

divisor_tensao({'tensao_entrada': 5, 'tensao_saida': 3.3})
  -> {"r1_ohms": 5600.0, "r2_ohms": 10000, "tensao_saida_real": 3.205}

largura_trilha({'corrente_a': 1.0})
  -> {"largura_mm": 0.3, "largura_mils": 11.8}

buscar_datasheet({'consulta': 'pinout', 'documento': 'arduino-uno'})
  -> erro: documento desconhecido: 'arduino-uno'. Use um destes: esp32-datasheet, dht22, hc-sr04, jlcpcb.

resistor_led

## Cliente

O cliente é um agente conectado ao servidor do grupo e a pelo menos um servidor MCP da comunidade, remoto ou local, cada um adicionado ao dicionário `mcpServers`. Com mais de um servidor, o `MCPAdapter` prefixa o nome de cada ferramenta com a chave do servidor, o que evita colisão quando dois servidores têm ferramentas com o mesmo nome. Cada ferramenta abre a conexão quando é chamada, e chamar o agente dentro de `async with adapter` faz as chamadas de uma pergunta compartilharem uma conexão.

In [7]:
servers = {
    "nosso": {"command": sys.executable, "args": ["server.py"]},
    # "time": {"command": "uvx", "args": ["mcp-server-time"]},
}

adapter = MCPAdapter({"mcpServers": servers})

O servidor da comunidade é o `@modelcontextprotocol/server-filesystem` (servidor de referência do MCP, rodado com `npx`, versão fixa), restrito à pasta `circuitos/`, onde ficam descrições de circuitos em Markdown. Com ele o agente lê o que o usuário deixou anotado sobre um projeto e usa as ferramentas do `bancada` para calcular.

In [8]:
servers = {
    "bancada": {"command": sys.executable, "args": ["server.py"]},
    "arquivos": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem@2.2.0", str(Path("circuitos").resolve())]},
}

adapter = MCPAdapter({"mcpServers": servers})

### Skill

A skill é uma pasta em `skills/` com um `SKILL.md`, e a pasta e o arquivo devem ser criados pela dupla: o frontmatter traz `name` e `description`, e o corpo traz as instruções. Ela deve ensinar uma tarefa do domínio que precise de vários passos, e não repetir o que uma ferramenta já faz sozinha:

- **Descrição:** diz o que a skill faz e quando usá-la, com as palavras que um usuário usaria no pedido.
- **Corpo:** indica quais ferramentas chamar e em que ordem, usando pelo menos duas do servidor da dupla, o que fazer quando uma falhar ou não encontrar nada e o formato exato da resposta. A skill pode combinar também ferramentas do servidor da comunidade. As ferramentas são citadas pelo nome que o agente vê, com a chave do servidor como prefixo, como `nosso_search_recipes`.

A ferramenta `load_skill` lê o `SKILL.md` pelo nome da skill, e o catálogo, que vai na mensagem de sistema, inclui o nome e a descrição de todas as skills da pasta `skills/`.

In [9]:
@tool
def load_skill(name: str) -> str:
    """Carrega as instruções completas de uma skill pelo nome."""
    return Path("skills", name, "SKILL.md").read_text()


metadata = [yaml.safe_load(path.read_text().split("---")[1]) for path in sorted(Path("skills").glob("*/SKILL.md"))]
catalog = "\n".join(f"- {meta['name']}: {meta['description']}" for meta in metadata)
print(catalog)

- ligar-componente-no-esp32: Planeja a ligação de um sensor, LED ou módulo num ESP32, conferindo nos datasheets se pode ligar direto e calculando o resistor ou o divisor de tensão. Use quando o usuário perguntar como ligar, conectar ou montar um componente no ESP32, ou se um componente pode ir direto num pino.


### Catálogo

Tabela com o servidor, o nome, a descrição e os argumentos de cada ferramenta disponível, seguida de uma avaliação das descrições do servidor da comunidade: quais deixam claro quando a ferramenta deve ser usada e quais são ambíguas ou se sobrepõem às do servidor da dupla.

In [10]:
async with adapter:
    todas = await adapter.list_tools()

catalogo_ferramentas = pd.DataFrame([
    {"servidor": t.name.split("_")[0], "nome": t.name, "descrição": t.description.split("\n")[0], "argumentos": list(t.args)}
    for t in todas + [load_skill]
])
catalogo_ferramentas

,servidor,nome,descrição,argumentos
0,bancada,bancada_buscar_datasheet,Busca trechos nos datasheets indexados e devolve o texto com documento e página para citar.,"[consulta, documento, k]"
1,bancada,bancada_resistor_led,"Calcula o resistor em série de um LED, R = (tensão da fonte - queda do LED) / corrente.","[tensao_fonte, tensao_led, corrente_ma]"
2,bancada,bancada_divisor_tensao,"Dimensiona um divisor resistivo para baixar um sinal, Vsaída = Ventrada * R2 / (R1 + R2).","[tensao_entrada, tensao_saida, r2_ohms]"
3,bancada,bancada_largura_trilha,Largura mínima de uma trilha de placa de circuito impresso pela norma IPC-2221.,"[corrente_a, espessura_cobre_oz, elevacao_temperatura_c, camada]"
4,arquivos,arquivos_read_file,Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.,"[path, tail, head]"
5,arquivos,arquivos_read_text_file,Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be...,"[path, tail, head]"
6,arquivos,arquivos_read_media_file,Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file ty...,[path]
7,arquivos,arquivos_read_multiple_files,Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you need to analyze or compare multiple files....,[paths]
8,arquivos,arquivos_write_file,Create a new file or completely overwrite an existing file with new content. Use with caution as it will overwrite existing files without warning. Handles t...,"[path, content]"
9,arquivos,arquivos_append_file,"Append text to the end of a file, creating the file if it does not exist. The existing content is never changed or replaced. The text is appended exactly as...","[path, content]"


In [11]:
for t in todas:
    if t.name.startswith("arquivos_"):
        print(f"{t.name}: {t.description[:400]}\n")

arquivos_read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.

arquivos_read_text_file: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regard

arquivos_read_media_file: Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.

arquivos_read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you need to analyze or compare multiple files. Each file's conte

**Avaliação das descrições do servidor da comunidade (`arquivos`).**

- **Claras sobre quando usar:** `read_text_file` (um arquivo, com `head` e `tail` para ler só parte), `list_directory`, `search_files` (deixa explícito que procura pelo NOME e nunca dentro do arquivo, o que evita usá-la para buscar conteúdo), `copy_file` e `list_allowed_directories` (diz para chamar antes de acessar caminhos).
- **Ambíguas ou sobrepostas entre si:** `read_file` está marcada como DEPRECATED e faz o mesmo que `read_text_file`, mas continua oferecida e ocupa espaço no contexto. `list_directory`, `list_directory_with_sizes` e `directory_tree` respondem quase a mesma pergunta, e a escolha entre elas depende de detalhes (tamanhos, recursão). `read_multiple_files` se sobrepõe a chamar `read_text_file` várias vezes.
- **Falta informação:** nenhuma descrição diz se o `path` deve ser absoluto ou relativo à pasta permitida. Por isso, nas perguntas abaixo, o agente chama `list_allowed_directories` antes de ler, um passo a mais em cada pergunta.
- **Sobreposição com o nosso servidor:** nenhuma. O `bancada` consulta datasheets e calcula, e o `arquivos` lê e escreve arquivos. Os prefixos `bancada_` e `arquivos_` deixam isso claro no catálogo.

Num uso real, deixaríamos de fora as ferramentas de escrita (`write_file`, `edit_file`, `move_file`, `create_directory`), que o agente não precisa e que poderiam apagar as descrições dos circuitos.

### Agente

Agente criado com `create_agent`, com as ferramentas de todos os servidores mais `load_skill` e com o catálogo de skills na mensagem de sistema. Devem ser feitas pelo menos sete perguntas, mostrando para cada uma as ferramentas chamadas e a resposta:

- pelo menos duas que dependam do servidor da dupla;
- pelo menos uma que dependa do servidor da comunidade;
- uma que precise combinar ferramentas dos dois servidores;
- uma que deva carregar a skill e uma parecida que não deva;
- uma que nenhum servidor responda.

In [12]:
async with adapter:
    tools = await adapter.list_tools()

# system_prompt = ""

# agent = create_agent(model, tools=tools + [load_skill], system_prompt=system_prompt)

In [13]:
system_prompt = f"""Você é o Assistente de Bancada: ajuda a ligar componentes num ESP32 e a preparar placas. Responda em português.

Ferramentas: as do servidor bancada consultam os datasheets indexados (ESP32, DHT22, HC-SR04, JLCPCB) e fazem cálculos; as do servidor arquivos leem as descrições de circuitos na pasta circuitos/. Todo valor técnico da resposta vem de uma ferramenta, com a fonte citada. Se nenhuma ferramenta responder o pedido, diga que não consegue ajudar com isso.

Skills: quando o pedido se encaixar na descrição de uma skill, chame load_skill(nome) antes de começar e siga as etapas dela. Perguntas rápidas sobre um valor não precisam de skill.

{catalog}"""

agent = create_agent(model, tools=tools + [load_skill], system_prompt=system_prompt)


def trajetoria(mensagens):
    passos = []
    for m in mensagens:
        if isinstance(m, AIMessage):
            passos += [f"{c['name']}({json.dumps(c['args'], ensure_ascii=False)[:150]})" for c in m.tool_calls]
    return passos


async def perguntar(pergunta):
    async with adapter:
        r = await agent.ainvoke({"messages": [HumanMessage(pergunta)]})
    passos = trajetoria(r["messages"])
    print(f"PERGUNTA: {pergunta}\nFERRAMENTAS: {passos or 'nenhuma'}\nRESPOSTA:\n{r['messages'][-1].text}\n" + "-" * 100)
    return {"pergunta": pergunta, "ferramentas": [p.split("(")[0] for p in passos], "resposta": r["messages"][-1].text}

In [14]:
PERGUNTAS = [
    ("servidor da dupla", "Quanta corrente um pino GPIO do ESP32 consegue fornecer?"),
    ("servidor da dupla", "Que resistor eu uso num LED azul de 3,0 V com 5 mA, ligado num GPIO de 3,3 V?"),
    ("servidor da comunidade", "Quais descrições de circuito existem na pasta circuitos?"),
    ("os dois servidores", "Leia a descrição do painel de status da estufa e calcule o resistor de cada LED."),
    ("deve carregar a skill", "Como eu ligo o sensor ultrassônico do medidor de nível da caixa d'água no ESP32? A descrição está na pasta circuitos."),
    ("não deve carregar a skill", "Qual é a tensão de trabalho do HC-SR04?"),
    ("nenhum servidor responde", "Qual vai ser a cotação do dólar amanhã?"),
]

resultados = []
for tipo, pergunta in PERGUNTAS:
    resultados.append({"tipo": tipo} | await perguntar(pergunta))

Key 'additionalProperties' is not supported in schema, ignoring


Key 'additionalProperties' is not supported in schema, ignoring


Key 'additionalProperties' is not supported in schema, ignoring


Key 'additionalProperties' is not supported in schema, ignoring


Key '$schema' is not supported in schema, ignoring


Key '$schema' is not supported in schema, ignoring


... (498 outputs left out)

Key '$schema' is not supported in schema, ignoring


Key '$schema' is not supported in schema, ignoring


/home/jcassio-dev/www/labs-unidade2/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


PERGUNTA: Qual vai ser a cotação do dólar amanhã?
FERRAMENTAS: nenhuma
RESPOSTA:
Desculpe, não consigo ajudar com isso. Minhas ferramentas são voltadas exclusivamente para auxiliar na ligação de componentes eletrônicos a um ESP32, consulta de datasheets (ESP32, DHT22, HC-SR04, JLCPCB) e cálculos de bancada, como resistores para LEDs e divisores de tensão.
----------------------------------------------------------------------------------------------------


In [15]:
pd.DataFrame(resultados)[["tipo", "pergunta", "ferramentas"]]

,tipo,pergunta,ferramentas
0,servidor da dupla,Quanta corrente um pino GPIO do ESP32 consegue fornecer?,"[bancada_buscar_datasheet, bancada_buscar_datasheet]"
1,servidor da dupla,"Que resistor eu uso num LED azul de 3,0 V com 5 mA, ligado num GPIO de 3,3 V?",[bancada_resistor_led]
2,servidor da comunidade,Quais descrições de circuito existem na pasta circuitos?,"[arquivos_list_allowed_directories, arquivos_list_directory]"
3,os dois servidores,Leia a descrição do painel de status da estufa e calcule o resistor de cada LED.,"[arquivos_list_directory, arquivos_list_allowed_directories, arquivos_search_files, arquivos_read_text_file, bancada_resistor_led, bancada_resistor_led, ban..."
4,deve carregar a skill,Como eu ligo o sensor ultrassônico do medidor de nível da caixa d'água no ESP32? A descrição está na pasta circuitos.,"[load_skill, arquivos_search_files, arquivos_list_allowed_directories, arquivos_list_directory, arquivos_read_text_file, bancada_buscar_datasheet, bancada_b..."
5,não deve carregar a skill,Qual é a tensão de trabalho do HC-SR04?,[bancada_buscar_datasheet]
6,nenhum servidor responde,Qual vai ser a cotação do dólar amanhã?,[]


### Análise das perguntas

| pergunta | o que se esperava | o que aconteceu |
|---|---|---|
| corrente do GPIO | servidor da dupla | duas buscas no `bancada_buscar_datasheet` (a segunda reformulada), resposta com 40 mA, 20 mA e 28 mA da tabela da p. 52, citada. **Porém** a resposta acrescenta uma "nota prática" (manter abaixo de 12 a 20 mA) que não veio de nenhuma ferramenta, e não cita a queda para 29 mA da nota de rodapé. |
| resistor do LED azul | servidor da dupla | uma chamada a `bancada_resistor_led` com os valores certos: 68 Ω comercial, 4,41 mA. |
| arquivos da pasta | servidor da comunidade | `list_allowed_directories` e `list_directory`, lista certa dos dois arquivos. |
| painel de status | os dois servidores | leu o arquivo com o servidor da comunidade e chamou `bancada_resistor_led` três vezes, uma por LED, com os valores do arquivo (150 Ω, 150 Ω e 68 Ω). Gastou três chamadas para achar o caminho do arquivo antes de ler. |
| ligar o sensor do medidor de nível | deve carregar a skill | carregou `ligar-componente-no-esp32` e seguiu as etapas na ordem: leu o arquivo, fez as duas buscas da etapa 2, calculou o divisor e respondeu no formato da skill (tabela, componentes extras, fontes). Uma imprecisão: para o limite do GPIO, citou a tabela de máximos absolutos da alimentação (p. 51) em vez do VIH da p. 52; a conclusão, de que o Echo de 5 V precisa de divisor, está certa. |
| tensão do HC-SR04 | não deve carregar a skill | não carregou; uma busca e a resposta de 5 V, citando a p. 1. |
| cotação do dólar | nenhum servidor responde | recusou sem chamar ferramentas. |

O agente escolheu bem entre os servidores e entre usar ou não a skill, e todos os valores calculados vieram das ferramentas. A skill fez diferença na pergunta mais longa: sem ela, a pergunta parecida respondeu só o valor, e com ela o agente conferiu o limite do ESP32 e calculou o divisor sem que o usuário pedisse. Os dois problemas que vimos são de conteúdo, não de ferramenta: a recomendação de 12 a 20 mA sem fonte na primeira pergunta e a página trocada na resposta da skill. As respostas também usaram LaTeX (`$3,0\text{ V}$`), que a mensagem de sistema não proibiu.